# 1 load package and data`

In [ ]:
import json
from collections import namedtuple, Counter

import numpy as np
import pandas as pd
import scanpy as sc
import pandas as pd
import squidpy as sq

## Image manipulation and geometry
from tifffile import imread
from skimage.io import imread as skimread

## Plotting imports
from matplotlib import pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_hex, Colormap
from matplotlib.cm import ScalarMappable
from matplotlib.colorbar import ColorbarBase
from matplotlib.lines import Line2D
from matplotlib import rc_context

In [ ]:
folder_path = r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\analysis\2st_2025_4_9\backup_adata\QC_10_50_2st\embryo"
h5ad_file = Path(folder_path) / "whole_embryo_hvg_res_1_updata.h5ad"
embryo = sc.read_h5ad(h5ad_file)

# 2 process selected cluster of gut tube

In [ ]:
import numpy as np
keep = np.array(["e17_6"], dtype=str)

pgc= embryo[embryo.obs["leiden_hvg_sub"].isin(keep)].copy()

In [ ]:
if pgc.n_obs == 0:
    print("Warning: pgc is empty. Aborting processing.")
else:
    print(f"Info: pgc selected with {pgc.n_obs} cells.")

    if 'counts_raw' in pgc.layers:
        pgc.X = pgc.layers['counts_raw'].copy()
        print("Info: .X initialized from 'counts_raw' layer.")

    sc.pp.normalize_total(pgc, inplace=True)
    sc.pp.log1p(pgc)

    sc.pp.highly_variable_genes(pgc, n_top_genes=2000)

    pgc.raw = pgc.copy()

    sc.pp.scale(pgc, max_value=10)

    sc.tl.pca(pgc, use_highly_variable=True)

    sc.pp.neighbors(pgc)

    sc.tl.umap(pgc, min_dist=0.01, spread=2)

    sc.tl.leiden(pgc, resolution=0.5, key_added='leiden_sub')

    print(f"pgc (n={pgc.n_obs}) processing complete.")
    print(f"Total genes retained: {pgc.n_vars}")
    print(f"Highly variable genes used for analysis: {pgc.var.highly_variable.sum()}")

In [ ]:
sc.tl.rank_genes_groups(pgc, 
                        groupby="leiden_sub", 
                        method="wilcoxon", 
                        pts=True,
                        )

In [ ]:
deg_df = sc.get.rank_genes_groups_df(pgc, group=None)
deg_df.to_excel(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure3\DEG\DEG_leiden_hvg_sub_pgc.xlsx", index=True)

In [ ]:
sc.tl.dendrogram(gut, groupby="leiden_sub", linkage_method="ward",cor_method='pearson')  #'pearson', 'spearman', 'kendall',
sc.pl.dendrogram(gut, groupby="leiden_sub")

In [ ]:
pgc.write_h5ad(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\analysis\2st_2025_4_9\backup_adata\QC_10_50_2st\embryont_pgc.h5ad")

# 3 Figure 5a

In [ ]:
custom_palette= {
    "0": "#fb34cd",
    "1": "#FE664D",
    "2": "#009203"
}

In [ ]:

tsne_coords =pgc.obsm['X_umap']


leiden_clusters = pgc.obs['leiden_sub']

import pandas as pd

tsne_df = pd.DataFrame(tsne_coords, columns=['UMAP_1', 'UMAP_2'])
tsne_df['leiden_sub'] = leiden_clusters.values

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(4, 2.8))
sns.scatterplot(data=tsne_df, x='UMAP_1', y='UMAP_2', hue='leiden_sub', palette=custom_palette, s=35, edgecolor=None)
plt.title("UMAP Plot with Leiden Clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(title="Leiden Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\umap\PGC\umap_PGC.png', format="png",dpi=900,bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd

df = pd.read_csv(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\analysis\2st_2025_4_9\cellnest_data_from _alice\13st_2026_01_08_PGC_e17.4_e8.0\k50\CellNEST_MYDATA_3D_top20percent.csv")


m = embryo.obs.set_index('cell_id_1')['leiden_hvg_sub']


df['leiden_from'] = df['from_cell'].map(m)


df['leiden_to'] = df['to_cell'].map(m)

# 4 Figure 5b

In [ ]:
my_genes=[
     "POU5F1","MDM2","NEPRO","MGMT", "SKA3","CD99","COL2A1"
]
sc.pl.dotplot(
    pgc,
    var_names=my_genes,
    groupby='leiden_sub', 
    standard_scale='var',  
    #vmax=2, 
    cmap="Reds",
    figsize=(4.3, 1),
    dendrogram=True,
    save='pgc_leiden_sub_normalize.pdf'
)

# 5 Figure 5f

In [ ]:
def visualize_lr_network(df, from_leiden, to_leiden, top_n=15, 
                        show_scatter=False, save_path=None,
                        figsize=None, dpi=300, 
                        color_by='specificity'): 

    import matplotlib.pyplot as plt
    import seaborn as sns
    from adjustText import adjust_text
    

    pair_df = df[(df['leiden_from'] == from_leiden) & 
                 (df['leiden_to'] == to_leiden)].copy()
    
    if len(pair_df) == 0:

        return
    

    lr_summary = pair_df.groupby(['ligand', 'receptor']).agg({
        'attention_score': ['mean', 'count'],
        'specificity_score': 'mean'  
    }).reset_index()
    lr_summary.columns = ['ligand', 'receptor', 'avg_attention', 'count', 'avg_specificity']
    

    lr_summary = lr_summary.nlargest(top_n, 'count')
    
    if len(lr_summary) == 0:
    
        return
    
    lr_summary = lr_summary.sort_values('count', ascending=True)
    

    if figsize is None:
        if show_scatter:
            figsize = (18, 8)
        else:
            width = 10
            height = max(6, len(lr_summary) * 0.5)
            figsize = (width, height)
    

    if show_scatter:
        fig, axes = plt.subplots(1, 2, figsize=figsize)
        ax1 = axes[0]
    else:
        fig, ax1 = plt.subplots(figsize=figsize)
    
  
    lr_summary['pair'] = lr_summary['ligand'] + ' _ ' + lr_summary['receptor']
    
    if color_by == 'specificity':
      
        color_values = lr_summary['avg_specificity']
        cbar_label = 'Avg Specificity Score'
        global_min = lr_summary['avg_specificity'].min()
        global_max = lr_summary['avg_specificity'].max()
    elif color_by == 'attention':
       
        color_values = lr_summary['avg_attention']
        cbar_label = 'Avg Attention Score'
        global_min = lr_summary['avg_attention'].min()
        global_max = lr_summary['avg_attention'].max()
    else:
       
        color_values = None
    

    if color_values is not None:
        norm = plt.Normalize(vmin=global_min, vmax=global_max)
        cmap = plt.cm.RdYlBu_r  
        colors = cmap(norm(color_values))
        
        bars = ax1.barh(range(len(lr_summary)), lr_summary['count'], 
                        color=colors, edgecolor='black', linewidth=0.5)
        
        # 添加 colorbar
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        plt.colorbar(sm, ax=ax1, label=cbar_label)
    else:
        bars = ax1.barh(range(len(lr_summary)), lr_summary['count'], 
                        color='steelblue', edgecolor='black', linewidth=0.5)
    
    ax1.set_yticks(range(len(lr_summary)))
    ax1.set_yticklabels(lr_summary['pair'], fontsize=10)
    ax1.set_xlabel('Interaction Count', fontsize=12, fontweight='bold')
    ax1.set_title(f'Top {top_n} L-R Pairs: {from_leiden} → {to_leiden}', 
                  fontsize=14, fontweight='bold')
    #ax1.grid(axis='x', alpha=0.3)
    
  
    if show_scatter:
        ax2 = axes[1]
        
        if color_by == 'specificity':
            scatter = ax2.scatter(lr_summary['count'], lr_summary['avg_specificity'],
                                 s=200, c=lr_summary['avg_specificity'],
                                 cmap=cmap, norm=norm, alpha=0.8,
                                 edgecolors='black', linewidth=1)
            ax2.set_ylabel('Average Specificity Score', fontsize=12)
        else:
            scatter = ax2.scatter(lr_summary['count'], lr_summary['avg_attention'],
                                 s=200, color='steelblue', alpha=0.7,
                                 edgecolors='black', linewidth=1)
            ax2.set_ylabel('Average Attention Score', fontsize=12)
        
        ax2.set_xlabel('Interaction Count', fontsize=12)
        ax2.set_title('L-R Pair Frequency vs Specificity', fontsize=14)
        ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=dpi, bbox_inches='tight')

    
    plt.show()
    
    return lr_summary.sort_values('count', ascending=False)

In [ ]:
result = visualize_lr_network(df, 'e8_0', 'e17_6.2', 
                              top_n=15,
                              figsize=(3, 4),
                              min_attention=0.6,
                              color_by='none',
                              save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\4st_1st_revised_2026_03_04\figure 5\cellnest\cellnest_e8_0_e17_6.2.pdf')

In [ ]:

result = visualize_lr_network(df, 'e17_4', 'e17_6.2', 
                              top_n=15,
                              figsize=(3, 4),
                              min_attention=0.6,
                              color_by='none',
                              save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\4st_1st_revised_2026_03_04\figure 5\cellnest\cellnest_e17_4_e17_6.2.pdf')

# 6 Figure 5g,h

## prepare data

In [ ]:
import scanpy as sc
import pandas as pd
from pathlib import Path

folder_path = r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\analysis\2st_2025_4_9\backup_adata\QC_10_50_2st"
h5ad_file = Path(folder_path) / "final_mtx_v3_Fu_adjustedz_subcl.h5ad"
adata= sc.read_h5ad(h5ad_file)
adata

In [ ]:

adata.obs['leiden_hvg_1'] = None
adata.obs['source'] = None


embryo_mapping = embryo.obs['leiden_hvg_1']
mask_embryo = adata.obs.index.isin(embryo_mapping.index)
adata.obs.loc[mask_embryo, 'leiden_hvg_1'] = embryo_mapping.reindex(adata.obs.index[mask_embryo]).values
adata.obs.loc[mask_embryo, 'source'] = 'embryo'

placenta_mapping = placenta.obs['leiden_hvg_1']
mask_placenta = adata.obs.index.isin(placenta_mapping.index)
adata.obs.loc[mask_placenta, 'leiden_hvg_1'] = placenta_mapping.reindex(adata.obs.index[mask_placenta]).values
adata.obs.loc[mask_placenta, 'source'] = 'placenta'


In [ ]:
import numpy as np


adata.obs['cluster'] = 'other'

adata.obs.loc[(adata.obs['source'] == 'embryo') & (adata.obs['leiden_hvg_1'] == '7'), 'cluster'] = 'blood'
adata.obs.loc[(adata.obs['source'] == 'placenta') & (adata.obs['leiden_hvg_1'] == '20'), 'cluster'] = 'blood'


adata.obs.loc[(adata.obs['source'] == 'embryo') & (adata.obs['leiden_hvg_1'].isin(['9', '24'])), 'cluster'] = 'EC'
adata.obs.loc[(adata.obs['source'] == 'placenta') & (adata.obs['leiden_hvg_1'] == '10'), 'cluster'] = 'EC'


adata.obs.loc[(adata.obs['source'] == 'embryo') & (adata.obs['leiden_hvg_1'].isin(['13', '26'])), 'cluster'] = 'IM'
adata.obs.loc[(adata.obs['source'] == 'placenta') & (adata.obs['leiden_hvg_1'] == '22'), 'cluster'] = 'IM'


adata.obs.loc[(adata.obs['source'] == 'embryo') & (adata.obs['leiden_hvg_1'] == '21'), 'cluster'] = 'NC'


adata.obs.loc[(adata.obs['source'] == 'placenta') & (adata.obs['leiden_hvg_1'] == '1'), 'cluster'] = 'FB'


adata.obs.loc[(adata.obs['source'] == 'placenta') & (adata.obs['leiden_hvg_1'] == '14'), 'cluster'] = 'PER'



In [ ]:

adata.obs['leiden_hvg_sub'] = None


embryo_mapping = embryo.obs['leiden_hvg_sub']
mask_embryo = adata.obs.index.isin(embryo_mapping.index)
adata.obs.loc[mask_embryo, 'leiden_hvg_sub'] = embryo_mapping.reindex(adata.obs.index[mask_embryo]).values


placenta_mapping = placenta.obs['leiden_hvg_sub']
mask_placenta = adata.obs.index.isin(placenta_mapping.index)
adata.obs.loc[mask_placenta, 'leiden_hvg_sub'] = placenta_mapping.reindex(adata.obs.index[mask_placenta]).values

adata_filtered = adata[~adata.obs['cluster'].isin(['other', 'FB'])].copy()


adata.obs['leiden_hvg_sub'] = adata.obs['leiden_hvg_sub'].astype(str)

mask_placenta = adata.obs['source'] == 'placenta'
adata.obs.loc[mask_placenta, 'leiden_hvg_sub'] = 'p' + adata.obs.loc[mask_placenta, 'leiden_hvg_sub']


In [ ]:
import pandas as pd

df = pd.read_csv(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\From Jamilla\DASH_APP\13-2026-02-19_Figure5\selected_cells_2.csv")

In [ ]:
ec.obs.loc[ec.obs['source'] == 'BS', 'source'] = 'embryo'
adata.obs.loc[adata.obs['source'] == 'BS', 'source'] = 'embryo'
adata_filtered.obs.loc[adata_filtered.obs['source'] == 'BS', 'source'] = 'embryo'



adata.obs['source'] = adata.obs['source'].astype(str)


matching_cells = adata.obs.index.isin(df['cell_id'])
adata.obs.loc[matching_cells, 'source'] = 'BS'




adata_filtered.obs['source'] = adata_filtered.obs['source'].astype(str)


matching_cells = adata_filtered.obs.index.isin(df['cell_id'])
adata_filtered.obs.loc[matching_cells, 'source'] = 'BS'



In [ ]:
adata_filtered.write_h5ad(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\adata\EC_IM_BL_NC.h5ad")

## figure 0f EC

In [ ]:
keep = np.array(["EC"], dtype=str)

ec = adata_filtered[adata_filtered.obs["cluster"].isin(keep)].copy()

In [ ]:
ec.layers['counts_raw'] = ec.X.copy()

sc.pp.normalize_total(ec, inplace=True)
sc.pp.log1p(ec)


sc.pp.highly_variable_genes(ec, n_top_genes=2000)


ec.raw = ec.copy()


sc.pp.scale(ec, max_value=10)


sc.tl.pca(ec, use_highly_variable=True)


sc.pp.neighbors(ec)


sc.tl.umap(ec, min_dist=0.01, spread=2)


sc.tl.leiden(ec, resolution=0.5, key_added='leiden_sub')

print(f"ec (n={ec.n_obs}) processing complete.")
print(f"Total genes retained: {ec.n_vars}")
print(f"Highly variable genes used for analysis: {ec.var.highly_variable.sum()}")

In [ ]:
sc.tl.rank_genes_groups(ec, 
                        groupby="leiden_hvg_sub", 
                        method="wilcoxon", 
                        pts=True,
                        )

In [ ]:
deg_df = sc.get.rank_genes_groups_df(ec, group=None)

deg_df.to_excel(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\deg\DEG_leiden_hvg_sub_ec.xlsx", index=True)



In [ ]:
sc.tl.dendrogram(ec, groupby="leiden_hvg_sub", linkage_method="ward",cor_method='pearson')  #'pearson', 'spearman', 'kendall',

sc.pl.dendrogram(ec, groupby="leiden_hvg_sub")



In [ ]:
custom_palette = {

    'e9_0': '#fb34cd',  
    'e9_1': '#e5e022',  
    'e9_2': '#10686f',  
    'e9_3': '#ffa5aa',  
    'e9_4': '#C1A029',  
    'e9_5': '#ff002a', 
    

    'e24_0': '#0066FF',
    'e24_1': '#FF4500', 
    'e24_2': '#FFD700',  
    
   'p10_0': '#00FFFF',  
   'p10_1': '#7FFF00',  
   'p10_2': '#8A2BE2', 
   'p10_3': '#0066FF', 
}

In [ ]:

tsne_coords =ec.obsm['X_umap']


leiden_clusters = ec.obs['leiden_hvg_sub']

import pandas as pd

tsne_df = pd.DataFrame(tsne_coords, columns=['UMAP_1', 'UMAP_2'])
tsne_df['leiden_hvg_sub'] = leiden_clusters.values

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
sns.scatterplot(data=tsne_df, x='UMAP_1', y='UMAP_2', hue='leiden_hvg_sub', palette=custom_palette, s=5, edgecolor=None)
plt.title("UMAP Plot with Leiden Clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(title="Leiden Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\umap\umap_EC.png', format="png",dpi=900,bbox_inches='tight')
plt.show()

In [ ]:
my_genes=[
     "KDR","BMPER", "SLC9A3R1","SGK1","SOX17", "HMGA1","THBS1","CARTPT", "GJA5","PAGE4","HGF", "IGF2",
]
sc.pl.dotplot(
    ec,
    var_names=my_genes,
    groupby='leiden_hvg_sub',  # 你的分组列名
    standard_scale='var',  # 标准化
    #vmax=2, 
    cmap="Reds",
    figsize=(6, 3.5),
    dendrogram=True,
    save='ec_leiden_sub_normalize.pdf'
)

# 7 Figure 5k,i

In [ ]:
keep = np.array(["blood"], dtype=str)

bl = adata_filtered[adata_filtered.obs["cluster"].isin(keep)].copy()

In [ ]:
bl.layers['counts_raw'] = bl.X.copy()

sc.pp.normalize_total(bl, inplace=True)
sc.pp.log1p(bl)

sc.pp.highly_variable_genes(bl, n_top_genes=2000)

bl.raw = bl.copy()

sc.pp.scale(bl, max_value=10)

sc.tl.pca(bl, use_highly_variable=True)

sc.pp.neighbors(bl)

sc.tl.umap(bl, min_dist=0.01, spread=2)

sc.tl.leiden(bl, resolution=0.5, key_added='leiden_sub')
print(f"bl (n={bl.n_obs}) processing complete.")
print(f"Total genes retained: {bl.n_vars}")
print(f"Highly variable genes used for analysis: {bl.var.highly_variable.sum()}")

In [ ]:
sc.tl.rank_genes_groups(bl, 
                        groupby="leiden_hvg_sub", 
                        method="wilcoxon", 
                        pts=True,
                        )

In [ ]:
deg_df = sc.get.rank_genes_groups_df(bl, group=None)

deg_df.to_excel(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\deg\DEG_leiden_hvg_sub_bl.xlsx", index=True)



In [ ]:
sc.tl.dendrogram(bl, groupby="leiden_sub", linkage_method="ward",cor_method='pearson')  #'pearson', 'spearman', 'kendall',

sc.pl.dendrogram(bl, groupby="leiden_sub")

In [ ]:

tsne_coords =bl.obsm['X_umap']


leiden_clusters = bl.obs['leiden_hvg_sub']

import pandas as pd

tsne_df = pd.DataFrame(tsne_coords, columns=['UMAP_1', 'UMAP_2'])
tsne_df['leiden_hvg_sub'] = leiden_clusters.values

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
sns.scatterplot(data=tsne_df, x='UMAP_1', y='UMAP_2', hue='leiden_hvg_sub', palette=custom_palette, s=5, edgecolor=None)
plt.title("UMAP Plot with Leiden Clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(title="Leiden Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\umap\umap_BL.png', format="png",dpi=900,bbox_inches='tight')
plt.show()

In [ ]:
my_genes=[
     "HBZ","KDR", "FRZB","BTK","KEL","CCNB1"
]
sc.pl.dotplot(
    bl,
    var_names=my_genes,
    groupby='leiden_sub',  # 你的分组列名
    standard_scale='var',  # 标准化
    #vmax=2, 
    cmap="Reds",
    figsize=(4.2, 2),
    dendrogram=True,
    save='bl_leiden_sub_normalize.pdf'
)

In [ ]:

moran_df = adata_detailed.uns['moranI'].copy()
is_significant = moran_df['pval_norm_fdr_bh'] < 0.05


is_paracrine = moran_df['sender'] != moran_df['receiver']


filtered_results = moran_df[is_significant & is_paracrine & is_positive].copy()


genes = filtered_results.index.tolist()


In [ ]:
sub_cluster_palette = {
    "e6_2":  "#fb34cd",  
    "e6_3":  "#FE664D",  
    "e6_6":  "#009203",  
    "e6_7":  "#ff002a",  
    
    "e8":  "#FF6347", 
    "e8_1":  "#8e0119",  
    "e8_2":  "#C1A029",  
    "e8_4":  "#00CED1",  
    
    "e11":   "#565DFD",  
    
    "e16_0": "#565DFD",  
    "e16_1": "#FF0000",  
    "e16_2": "#FF1493", 
    
    "e17_0": "#e5e022",  
    "e17_1": "#00FFFF", 
    "e17_2": "#4fa9ff",  
    "e17_3": "#1068be",  
    "e17_4": "#565DFD", 
    "e17_5": "#ffa5aa",  
    "e17_6": "#ADFF2F", 
    "e17_6.0": "#fb34cd",
    "e17_6.1": "#FE664D",
    "e17_6.2": "#009203",
    "e17_7": "#66c102",  
    "e20_0": "#66c102",
    "e20_1": "#8e0119", 
    "e20_2": "#10686f",

  
    "e25_0": "#7FFF00", 
    "e25_1": "#9f50f9",
    "e25_2": "#32a88e" , 
    
   "e22_0": "#AB76AE",  
    "e22_1": "#6a32a8", 
    "e22_2": "#FF1493", 
    "e22_3": "#10686f",  
}

In [ ]:

wnt_genes = [g for g in genes if 'WNT5A_FZD3' in g or 'SHH_PTCH1' in g or 'BMP7_BMPR1B' in g]
norm_df = create_advanced_lr_heatmap(
    adata_detailed, 
    gene_list=genes, 
    axis_col='AP',
    sender_col='sender',     
    receiver_col='receiver',
    cluster_colors=sub_cluster_palette, 
    n_bins=40,
    figsize=(6, 4),        
    order_method='peak_grouped',
    binning='linear',
    vlines=[280,480,840, 2100,2600],
    smooth='gaussian', smooth_sigma=1.5,
    highlight_genes=wnt_genes, 
    highlight_label_position='right',
    vmin=-2, vmax=2,
    show_only_highlighted=True,
    #x_tick_interval=500,
    xlim=(0,3000),
    save_path=r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure4\cellnest\cellnest_GUT_DOSAL_niche_heatmap_along_ap_small_1.png'
)


In [ ]:
df.to_csv(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\From Jamilla\DASH_APP\13-2026-02-19_Figure5\df.csv", index=False)

# 8 Figure 5m,n

In [ ]:
keep = np.array(["IM"], dtype=str)

im = adata_filtered[adata_filtered.obs["cluster"].isin(keep)].copy()

In [ ]:
im.layers['counts_raw'] = im.X.copy()

sc.pp.normalize_total(im, inplace=True)
sc.pp.log1p(im)

sc.pp.highly_variable_genes(im, n_top_genes=2000)

im.raw = im.copy()

sc.pp.scale(im, max_value=10)

sc.tl.pca(im, use_highly_variable=True)

sc.pp.neighbors(im)

sc.tl.umap(im, min_dist=0.01, spread=2)

sc.tl.leiden(im, resolution=0.5, key_added='leiden_sub')
print(f"im (n={im.n_obs}) processing complete.")
print(f"Total genes retained: {im.n_vars}")
print(f"Highly variable genes used for analysis: {im.var.highly_variable.sum()}")

In [ ]:
sc.tl.rank_genes_groups(im, 
                        groupby="leiden_sub", 
                        method="wilcoxon", 
                        pts=True,
                        )

In [ ]:
deg_df = sc.get.rank_genes_groups_df(im, group=None)

deg_df.to_excel(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\deg\DEG_leiden_hvg_sub_im.xlsx", index=True)



In [ ]:
sc.tl.dendrogram(im, groupby="leiden_sub", linkage_method="ward",cor_method='pearson')  #'pearson', 'spearman', 'kendall',

sc.pl.dendrogram(im, groupby="leiden_sub")

In [ ]:
custom_palette = {
    '0': '#00FF7F',  
    '1': '#FF1493',
    '2': '#FF8C00', 
    '3': '#4169E1', 
    '4': '#FF00FF',  
}

In [ ]:

tsne_coords =imc.obsm['X_umap']


leiden_clusters = imc.obs['leiden_sub']

import pandas as pd

tsne_df = pd.DataFrame(tsne_coords, columns=['UMAP_1', 'UMAP_2'])
tsne_df['leiden_sub'] = leiden_clusters.values

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
sns.scatterplot(data=tsne_df, x='UMAP_1', y='UMAP_2', hue='leiden_sub', palette=custom_palette, s=5, edgecolor=None)
plt.title("UMAP Plot with Leiden Clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(title="Leiden Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\umap\im\umap_im.png', format="png",dpi=900,bbox_inches='tight')
plt.show()

In [ ]:
my_genes= [
   "MRC1","CD68", "F13A1", 
    "CCNB1", "PCNA", "OLFML3"
]
sc.pl.dotplot(
    imc,
    var_names=my_genes,
    groupby='leiden_sub',  # 你的分组列名
    standard_scale='var',  # 标准化
    #vmax=2, 
    cmap="Reds",
    figsize=(4.5, 2),
    dendrogram=True,
    save='im_leiden_sub_normalize.pdf'
)

# 9 Figure 5q,r

In [ ]:
keep = np.array(["NC"], dtype=str)

nc = adata_filtered[adata_filtered.obs["cluster"].isin(keep)].copy()

In [ ]:
nc.layers['counts_raw'] = nc.X.copy()

sc.pp.normalize_total(nc, inplace=True)
sc.pp.log1p(nc)

sc.pp.highly_variable_genes(nc, n_top_genes=2000)

nc.raw = nc.copy()

sc.pp.scale(nc, max_value=10)

sc.tl.pca(nc, use_highly_variable=True)

sc.pp.neighbors(nc)

sc.tl.umap(nc, min_dist=0.01, spread=2)

sc.tl.leiden(nc, resolution=0.5, key_added='leiden_sub')
print(f"nc (n={nc.n_obs}) processing complete.")
print(f"Total genes retained: {nc.n_vars}")
print(f"Highly variable genes used for analysis: {nc.var.highly_variable.sum()}")

In [ ]:
sc.tl.rank_genes_groups(nc, 
                        groupby="leiden_sub", 
                        method="wilcoxon", 
                        pts=True,
                        )

In [ ]:
deg_df = sc.get.rank_genes_groups_df(nc, group=None)

deg_df.to_excel(r"P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\deg\DEG_leiden_hvg_sub_nc.xlsx", index=True)

In [ ]:
sc.tl.dendrogram(nc, groupby="leiden_sub", linkage_method="ward",cor_method='pearson')  #'pearson', 'spearman', 'kendall',

sc.pl.dendrogram(nc, groupby="leiden_sub")

In [ ]:
custom_palette = {
   '0': '#D2691E',  
    '1': '#9ACD32',  
    '2': '#00BFFF',  
    '3': '#66CDAA', 
}

In [ ]:

tsne_coords =nc.obsm['X_umap']


leiden_clusters = nc.obs['leiden_sub']

import pandas as pd
tsne_df = pd.DataFrame(tsne_coords, columns=['UMAP_1', 'UMAP_2'])
tsne_df['leiden_sub'] = leiden_clusters.values

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
sns.scatterplot(data=tsne_df, x='UMAP_1', y='UMAP_2', hue='leiden_sub', palette=custom_palette, s=8, edgecolor=None)
plt.title("UMAP Plot with Leiden Clusters")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(title="Leiden Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(r'P:\PI\PI_Chuva_de_Sousa_Lopes\susana\Fu\Project\20_Xenium 5k embryo\paper\3st_2025_1_27\Figure5\umap\nc\umap_nc.png', format="png",dpi=900,bbox_inches='tight')
plt.show()

In [ ]:
my_genes= [
   "SOX10","MPZ","TWIST1", "IRX1", 
    "PCDH8", "SEMA3C"
]
sc.pl.dotplot(
    nc,
    var_names=my_genes,
    groupby='leiden_sub',  
    standard_scale='var',  
    #vmax=2, 
    cmap="Reds",
    figsize=(4.2, 2),
    dendrogram=True,
    save='nc_leiden_sub_normalize.pdf'
)